# Feature Engineering

This notebook creates meaningful features from the cleaned dataset to improve predictive performance while avoiding data leakage. The engineered dataset will be used in preprocessing and model training.

In [1]:
# Import the required libraries for feature engineering.
import pandas as pd
import numpy as np

In [2]:
# Load the cleaned dataset from the processed data folder.
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [3]:
# Display the first few rows to verify that the dataset loaded correctly.
df.head()

,Date,System,Control,Type,Air temperature (K),Process temperature (K),Rotational speed (rpm),Torque (Nm),Tool wear (min),Diagnostic
0,2014-04-15 11:56:00,0,C,M,300.064358,310.033081,1541.242402,42.8,0.00000,No failure
1,2014-04-12 16:09:00,0,A,L,298.200000,308.700000,1408.000000,39.9,110.52424,No failure
2,2014-04-13 01:13:00,0,A,L,298.100000,308.500000,1498.000000,39.9,110.52424,No failure
3,2014-07-24 20:35:00,0,B,L,300.064358,310.033081,1433.000000,39.5,110.52424,No failure
4,2014-07-22 01:31:00,0,C,L,300.064358,310.033081,1541.242402,40.0,9.00000,No failure


In [4]:
# Check the dataset structure and data types before creating new features.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Date                     10000 non-null  str    
 1   System                   10000 non-null  int64  
 2   Control                  10000 non-null  str    
 3   Type                     10000 non-null  str    
 4   Air temperature (K)      10000 non-null  float64
 5   Process temperature (K)  10000 non-null  float64
 6   Rotational speed (rpm)   10000 non-null  float64
 7   Torque (Nm)              10000 non-null  float64
 8   Tool wear (min)          10000 non-null  float64
 9   Diagnostic               10000 non-null  str    
dtypes: float64(5), int64(1), str(4)
memory usage: 781.4 KB


In [5]:
# Convert the Date column to datetime format for time-based feature extraction.
df['Date'] = pd.to_datetime(df['Date']) 

In [6]:
# Verify that the Date column has been converted successfully.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Date                     10000 non-null  datetime64[us]
 1   System                   10000 non-null  int64         
 2   Control                  10000 non-null  str           
 3   Type                     10000 non-null  str           
 4   Air temperature (K)      10000 non-null  float64       
 5   Process temperature (K)  10000 non-null  float64       
 6   Rotational speed (rpm)   10000 non-null  float64       
 7   Torque (Nm)              10000 non-null  float64       
 8   Tool wear (min)          10000 non-null  float64       
 9   Diagnostic               10000 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(3)
memory usage: 781.4 KB


In [7]:
# Extract the year from the Date column.
df['Year'] = df['Date'].dt.year

In [8]:
# Extract the month from the Date column.
df["Month"] = df["Date"].dt.month

In [9]:
# Extract the day of the month from the Date column.
df["Day"] = df["Date"].dt.day

In [10]:
# Extract the day of the week from the Date column.
df["Day_of_Week"] = df["Date"].dt.dayofweek

In [11]:
# Extract the quarter from the Date column.
df["Quarter"] = df["Date"].dt.quarter

In [12]:
# Remove the original Date column after extracting all useful time-based features.
df.drop(columns=["Date"], inplace=True)

# Verify that the Date column has been removed.
df.columns

Index(['System', 'Control', 'Type', 'Air temperature (K)',
       'Process temperature (K)', 'Rotational speed (rpm)', 'Torque (Nm)',
       'Tool wear (min)', 'Diagnostic', 'Year', 'Month', 'Day', 'Day_of_Week',
       'Quarter'],
      dtype='str')

## Time-Based Feature Engineering

The `Date` column contains valuable temporal information that cannot be used directly by most machine learning algorithms. Therefore, it is first converted into a datetime format, allowing the extraction of meaningful time-based features.

The following features are extracted:

- **Year:** Captures long-term trends or changes across different years.
- **Month:** Helps identify seasonal or monthly operational patterns.
- **Day:** Represents the day of the month, which may capture periodic maintenance or production cycles.
- **Day of Week:** Identifies weekly operational patterns, such as weekday and weekend differences.
- **Quarter:** Represents quarterly periods, which may reflect business cycles or scheduled maintenance activities.

These engineered features may improve the model's ability to learn temporal patterns associated with machine failures.

In [13]:
# Create a feature representing the difference between process and air temperature.
df["Temperature_Difference"] = (
    df["Process temperature (K)"] -
    df["Air temperature (K)"]
)

In [14]:
# Display the temperature-related columns to verify the engineered feature.
df[
    [
        "Air temperature (K)",
        "Process temperature (K)",
        "Temperature_Difference",
    ]
].head()

,Air temperature (K),Process temperature (K),Temperature_Difference
0,300.064358,310.033081,9.968723
1,298.200000,308.700000,10.500000
2,298.100000,308.500000,10.400000
3,300.064358,310.033081,9.968723
4,300.064358,310.033081,9.968723


### Observation

- The `Temperature_Difference` feature was successfully created by subtracting the air temperature from the process temperature.
- The calculated values are approximately **10 K** for the first few observations, indicating that the machine's internal operating temperature is consistently higher than the surrounding air temperature.
- This engineered feature provides a direct measure of thermal stress, which may help the machine learning model identify conditions associated with machine failures.

In [15]:
# Create a Power Index feature by multiplying torque and rotational speed.
df["Power_Index"] = (
    df["Torque (Nm)"] *
    df["Rotational speed (rpm)"]
)

In [16]:
# Display the columns used to create the Power Index feature.
df[
    [
        "Rotational speed (rpm)",
        "Torque (Nm)",
        "Power_Index"
    ]
].head()

,Rotational speed (rpm),Torque (Nm),Power_Index
0,1541.242402,42.8,65965.174787
1,1408.000000,39.9,56179.200000
2,1498.000000,39.9,59770.200000
3,1433.000000,39.5,56603.500000
4,1541.242402,40.0,61649.696062


### Observation

- The `Power_Index` feature was successfully created by multiplying torque and rotational speed.
- The feature combines two important operating parameters into a single indicator of mechanical load.
- Higher values of the Power Index may represent increased mechanical stress on the machine and could help improve the model's ability to detect failure conditions.

In [17]:
# Display the first five rows of the engineered dataset.
df.head()

,System,Control,Type,Air temperature (K),Process temperature (K),Rotational speed (rpm),Torque (Nm),Tool wear (min),Diagnostic,Year,Month,Day,Day_of_Week,Quarter,Temperature_Difference,Power_Index
0,0,C,M,300.064358,310.033081,1541.242402,42.8,0.00000,No failure,2014,4,15,1,2,9.968723,65965.174787
1,0,A,L,298.200000,308.700000,1408.000000,39.9,110.52424,No failure,2014,4,12,5,2,10.500000,56179.200000
2,0,A,L,298.100000,308.500000,1498.000000,39.9,110.52424,No failure,2014,4,13,6,2,10.400000,59770.200000
3,0,B,L,300.064358,310.033081,1433.000000,39.5,110.52424,No failure,2014,7,24,3,3,9.968723,56603.500000
4,0,C,L,300.064358,310.033081,1541.242402,40.0,9.00000,No failure,2014,7,22,1,3,9.968723,61649.696062


In [18]:
# Display the column names of the engineered dataset.
df.columns

Index(['System', 'Control', 'Type', 'Air temperature (K)',
       'Process temperature (K)', 'Rotational speed (rpm)', 'Torque (Nm)',
       'Tool wear (min)', 'Diagnostic', 'Year', 'Month', 'Day', 'Day_of_Week',
       'Quarter', 'Temperature_Difference', 'Power_Index'],
      dtype='str')

In [19]:
# Display dataset information after feature engineering.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   System                   10000 non-null  int64  
 1   Control                  10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature (K)      10000 non-null  float64
 4   Process temperature (K)  10000 non-null  float64
 5   Rotational speed (rpm)   10000 non-null  float64
 6   Torque (Nm)              10000 non-null  float64
 7   Tool wear (min)          10000 non-null  float64
 8   Diagnostic               10000 non-null  str    
 9   Year                     10000 non-null  int32  
 10  Month                    10000 non-null  int32  
 11  Day                      10000 non-null  int32  
 12  Day_of_Week              10000 non-null  int32  
 13  Quarter                  10000 non-null  int32  
 14  Temperature_Difference   10000 non

In [20]:
# Save the engineered dataset for the preprocessing stage.
df.to_csv("../data/processed/engineered_data.csv", index=False)

In [21]:
# Display the shape of the engineered dataset.
df.shape

(10000, 16)

In [22]:
# Check for missing values in the engineered dataset.
df.isnull().sum()

System                     0
Control                    0
Type                       0
Air temperature (K)        0
Process temperature (K)    0
Rotational speed (rpm)     0
Torque (Nm)                0
Tool wear (min)            0
Diagnostic                 0
Year                       0
Month                      0
Day                        0
Day_of_Week                0
Quarter                    0
Temperature_Difference     0
Power_Index                0
dtype: int64

In [23]:
# Check for duplicate rows in the engineered dataset.
df.duplicated().sum()

np.int64(1)

### Final Observation

- Time-based and domain-specific features were successfully engineered.
- The dataset structure remains consistent, with no unexpected missing values or duplicate records introduced during feature engineering.
- The engineered dataset has been saved and is now ready for the preprocessing stage of the machine learning pipeline.